# Ridge regression: five key points

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

## 1 and 2. Coefficients shrink, never to 0; the largest shrink most

In [2]:
data = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=2)
rows = {}
for alpha in [0, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]:
    reg = Ridge(alpha=alpha).fit(X_train, y_train)
    rows[alpha] = list(reg.coef_.round(2)) + [round(reg.score(X_test, y_test), 3)]
pd.DataFrame(rows, index=data.feature_names + ["test R2"]).T

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,test R2
0.000,-9.16,-205.45,516.69,340.62,-895.55,561.22,153.89,126.73,861.13,52.42,0.440
0.001,-8.76,-204.31,518.38,339.97,-787.70,475.28,106.80,114.63,819.75,52.87,0.441
0.010,-6.40,-198.66,522.06,336.34,-383.72,152.67,-66.05,75.61,659.88,55.83,0.444
0.100,6.64,-172.24,485.53,314.68,-72.94,-80.59,-174.46,83.61,484.37,73.59,0.452
1.000,42.24,-57.30,282.17,198.06,14.36,-22.55,-136.93,102.02,260.10,98.55,0.408
10.000,21.17,1.66,63.66,48.49,18.42,12.88,-38.92,38.84,61.61,35.51,0.152
100.000,2.86,0.63,7.54,5.85,2.71,2.14,-4.83,5.11,7.45,4.58,0.010
1000.000,0.30,0.07,0.77,0.60,0.28,0.23,-0.50,0.53,0.76,0.47,-0.010
10000.000,0.03,0.01,0.08,0.06,0.03,0.02,-0.05,0.05,0.08,0.05,-0.012


## 3. Bias and variance against alpha (fresh noise, known true curve)

In [3]:
# Toy data, because bias and variance need the true curve. 15 fixed training x, fresh noise for each of 300 draws.
f = lambda x: 0.7 * x ** 2 - 2 * x + 3
SD = 2.0
x_train = np.linspace(-2, 3, 15)[:, None]
x_test = (x_train[1:] + x_train[:-1]) / 2          # test x between the training x

def bias_variance(alpha, draws=300):
    r = np.random.default_rng(1)
    preds = []
    for _ in range(draws):
        y = f(x_train[:, 0]) + SD * r.standard_normal(len(x_train))
        model = make_pipeline(PolynomialFeatures(degree=15), StandardScaler(), Ridge(alpha=alpha))
        preds.append(model.fit(x_train, y).predict(x_test))
    preds = np.array(preds)
    bias2 = ((preds.mean(0) - f(x_test[:, 0])) ** 2).mean()
    var = preds.var(0).mean()
    return bias2 + var + SD ** 2, bias2, var

for alpha in [0.0001, 0.01, 1, 10, 100, 1000]:
    loss, bias2, var = bias_variance(alpha)
    print(f"alpha {alpha:>7}: error {loss:5.2f}  bias2 {bias2:5.3f}  variance {var:5.2f}")

alpha  0.0001: error  6.91  bias2 0.004  variance  2.91


alpha    0.01: error  5.58  bias2 0.004  variance  1.58


alpha       1: error  4.91  bias2 0.092  variance  0.82


alpha      10: error  5.90  bias2 1.426  variance  0.48


alpha     100: error  8.48  bias2 4.166  variance  0.31


alpha    1000: error  9.31  bias2 5.053  variance  0.26


## 4. The lowest point of the loss moves towards 0

In [4]:
X, y = make_regression(n_samples=100, n_features=1, n_informative=1, noise=20, random_state=13)
x = X.ravel()
for lam in [0, 10, 50, 100]:
    best_m = (x * (y + 2.29)).sum() / ((x * x).sum() + lam)    # intercept held at -2.29
    print(f"lambda {lam:3d}: lowest loss at m = {best_m:.2f}")

lambda   0: lowest loss at m = 27.83
lambda  10: lowest loss at m = 24.96
lambda  50: lowest loss at m = 17.69
lambda 100: lowest loss at m = 12.96
